# Notebook 4 — Joint-Schedule FT on ROCOv2 (Colab A100)

**Purpose:** Same Joint-Schedule FT control as Notebook 3 (UICD), now for
ROCOv2. Same two-phase LR schedule as Staged FT (5e-5 for 2 epochs, then
1e-5 for 3 epochs) but with **no freezing** -- all parameters trainable
throughout.

**Environment note:** your original `ROCOv2_DAMF_Multiseed.ipynb` used an
**unpinned** `pip install transformers peft ...` -- the exact original
versions are not recoverable. This notebook pins to `transformers==4.41.2`,
`peft==0.11.1` (the same versions validated for your UICD and RSICD work)
so that this new experiment is at least internally consistent with the
other new baselines. This is a genuine limitation to note in the paper --
see the note at the end of this notebook.

**Data source:** `eltorio/ROCOv2-radiology` from HuggingFace (same as your
original ROCOv2 training).

**Dataset-specific config (preserved from your original notebook):**
- `max_length = 40` (not 30 -- radiology captions run longer)
- Single reference caption per image (not multi-reference like UICD/RSICD)
- `batch_size = 16`

**Runtime:** ROCOv2 is the largest dataset (~60K train images) -- expect
~4-6 hours per seed on A100, ~15-20 hours total for 3 seeds. Use background
execution; this will span multiple sessions.

**Output:** `joint_schedule_ft_rocov2_seed{42,0,123}.json` in
`/content/drive/MyDrive/DAMF/logs/` (consolidated with your other logs,
not the separate `DAMF_ROCOv2_checkpoints/` folder your original notebook
used -- keeping everything in one place going forward).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install pinned dependencies

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg,
                    '-q', '--no-warn-script-location'], check=True)

pip_install('transformers==4.41.2')
pip_install('peft==0.11.1')
pip_install('pycocoevalcap')
pip_install('nltk')
pip_install('datasets')
pip_install('evaluate')

print('Installations complete. RESTART RUNTIME NOW (Runtime -> Restart runtime),')
print('then re-run cells 1 and 2 before proceeding to cell 3.')

## IMPORTANT -- restart runtime once

After Cell 2 completes, click **Runtime -> Restart runtime**, then re-run cells 1 and 2. Verify with the next cell.

In [ ]:
import transformers, peft
print(f'transformers : {transformers.__version__}  (need 4.41.2)')
print(f'peft         : {peft.__version__}  (need 0.11.1)')
assert transformers.__version__ == '4.41.2', \
    f'STOP: transformers is {transformers.__version__}. Restart runtime and re-run cell 2.'
assert peft.__version__ == '0.11.1', \
    f'STOP: peft is {peft.__version__}. Restart runtime and re-run cell 2.'
print('Versions verified.')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
print('NLTK data downloaded.')

## 3. Imports and environment report

In [ ]:
import sys, os, gc, json, math, random
import numpy as np
import torch
import shutil as _shutil

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
from datasets import load_dataset

from transformers import BlipProcessor, BlipForConditionalGeneration
from peft import LoraConfig, get_peft_model
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

print('=' * 55)
print('ENVIRONMENT REPORT')
print('=' * 55)
print(f'Python       : {sys.version.split()[0]}')
print(f'PyTorch      : {torch.__version__}')
print(f'Transformers : {transformers.__version__}')
print(f'PEFT         : {peft.__version__}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Active device: {DEVICE}')

## 4. Configuration (preserved from your original ROCOv2 notebook)

Note `max_length=40`, different from UICD/RSICD's 30 -- radiology captions run longer.

In [ ]:
OUT = '/content/drive/MyDrive/DAMF/logs'
os.makedirs(OUT, exist_ok=True)

SEEDS = [42, 0, 123]

CFG = {
    'batch_size': 16,
    'max_length': 40,   # radiology captions run longer than UICD/RSICD
    'beam_size': 3, 'num_workers': 2,
    'stage1_epochs': 2, 'stage2_epochs': 3, 'total_naive_epochs': 5,
    'lr_naive': 1e-4, 'lr_lowlr': 1e-5, 'lr_stage1': 5e-5, 'lr_stage2': 1e-5,
    'lr_lora': 1e-4, 'weight_decay': 0.01,
    'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
    'rt_log_every_n_steps': 10, 'rt_epsilon': 1e-8,
}

print(f'CFG loaded. Output directory: {OUT}')

## 5. GradientTracker (verbatim from your UICD/RSICD pipeline)

In [ ]:
class GradientTracker:
    def __init__(self, model, model_type='blip'):
        self.eps = CFG['rt_epsilon']
        self.model_type = model_type
        self._vis_norms = []
        self._lang_norms = []
        self._hooks = []

        if model_type == 'blip':
            vis_key, lang_key = 'vision_model', 'text_decoder'
        elif model_type == 'blip2':
            vis_key, lang_key = 'vision_model', 'language_model'
        else:
            raise ValueError(f'Unknown model_type: {model_type}')

        n_vis, n_lang, n_skipped = 0, 0, 0
        for name, param in model.named_parameters():
            if not param.requires_grad:
                n_skipped += 1
                continue
            if vis_key in name:
                def make_vis_hook(n=name):
                    def hook(grad):
                        if grad is not None:
                            self._vis_norms.append(grad.detach().norm().item())
                    return hook
                self._hooks.append(param.register_hook(make_vis_hook()))
                n_vis += 1
            elif lang_key in name:
                def make_lang_hook(n=name):
                    def hook(grad):
                        if grad is not None:
                            self._lang_norms.append(grad.detach().norm().item())
                    return hook
                self._hooks.append(param.register_hook(make_lang_hook()))
                n_lang += 1

        print(f'  GradientTracker hooked: visual={n_vis}, language={n_lang}, frozen={n_skipped}')
        if n_vis == 0:
            print('  WARNING: No visual parameters hooked.')
        if n_lang == 0:
            print('  WARNING: No language parameters hooked.')

    def get_rt(self):
        if not self._vis_norms or not self._lang_norms:
            return None
        nv = math.sqrt(sum(v ** 2 for v in self._vis_norms))
        nl = math.sqrt(sum(v ** 2 for v in self._lang_norms))
        return nl / (nv + self.eps)

    def reset(self):
        self._vis_norms.clear()
        self._lang_norms.clear()

    def remove(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()
        print('  GradientTracker hooks removed.')

print('GradientTracker defined.')

## 6. Training utilities (verbatim from your UICD/RSICD pipeline)

In [ ]:
def compute_bleu4(predictions, references):
    smoother = SmoothingFunction().method4
    pred_tokens = [p.split() for p in predictions]
    ref_tokens = [[r.split() for r in refs] for refs in references]
    return corpus_bleu(ref_tokens, pred_tokens, smoothing_function=smoother)

def compute_cider(predictions, references):
    try:
        from pycocoevalcap.cider.cider import Cider
        from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
        gts = {i: [{'caption': r} for r in refs] for i, refs in enumerate(references)}
        res = {i: [{'caption': p}] for i, p in enumerate(predictions)}
        tok = PTBTokenizer()
        sc, _ = Cider().compute_score(tok.tokenize(gts), tok.tokenize(res))
        return float(sc)
    except Exception as e:
        print(f'  CIDEr error: {e}')
        return None

def compute_meteor(predictions, references):
    try:
        import evaluate as hf_evaluate
        meteor = hf_evaluate.load('meteor')
        flat_refs = [refs[0] for refs in references]
        result = meteor.compute(predictions=predictions, references=flat_refs)
        return float(result['meteor'])
    except Exception as e:
        print(f'  METEOR error: {e}')
        return None

def get_val_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            inputs = {
                'pixel_values': batch['pixel_values'].to(DEVICE),
                'input_ids': batch['input_ids'].to(DEVICE),
                'attention_mask': batch['attention_mask'].to(DEVICE),
            }
            inputs['labels'] = inputs['input_ids'].clone()
            with torch.amp.autocast('cuda'):
                total += model(**inputs).loss.item()
            n += 1
    return total / max(n, 1)

def train_one_epoch(model, loader, optimizer, scaler, tracker=None, step_counter=None):
    model.train()
    total_loss, n_batches, rt_log = 0.0, 0, []
    if step_counter is None:
        step_counter = [0]
    for batch in loader:
        inputs = {
            'pixel_values': batch['pixel_values'].to(DEVICE),
            'input_ids': batch['input_ids'].to(DEVICE),
            'attention_mask': batch['attention_mask'].to(DEVICE),
        }
        inputs['labels'] = inputs['input_ids'].clone()
        optimizer.zero_grad()
        if tracker is not None:
            tracker.reset()
        with torch.amp.autocast('cuda'):
            loss = model(**inputs).loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        n_batches += 1
        step_counter[0] += 1
        if tracker is not None and step_counter[0] % CFG['rt_log_every_n_steps'] == 0:
            rt = tracker.get_rt()
            if rt and not math.isinf(rt) and not math.isnan(rt):
                rt_log.append((step_counter[0], rt))
    return total_loss / max(n_batches, 1), rt_log

def save_logs(logs, filename):
    path = os.path.join(OUT, filename)
    with open(path, 'w') as f:
        json.dump(logs, f, indent=2)
    print(f'  Saved: {filename}')
    return path

print('Utilities defined.')

## 7. ROCOv2 dataset and data loading (verbatim from your original notebook)

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print('Loading ROCOv2 (full dataset) from HuggingFace...')
rocov2_raw = load_dataset('eltorio/ROCOv2-radiology')
print(f"ROCOv2: {len(rocov2_raw['train'])} train / "
      f"{len(rocov2_raw['validation'])} val / {len(rocov2_raw['test'])} test")

BLIP_PROCESSOR = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')

class ROCOv2Dataset(Dataset):
    """ROCOv2 has exactly ONE caption per image (unlike UICD/RSICD's
    multiple references). Wrapped as a single-item list at eval time
    for compatibility with the shared BLEU/CIDEr/METEOR pipeline."""
    def __init__(self, hf_split, processor, max_length):
        self.data = hf_split
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['image'].convert('RGB')
        caption = item['caption']
        inputs = self.processor(images=image, text=caption, return_tensors='pt',
                                padding='max_length', truncation=True,
                                max_length=self.max_length)
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'caption': caption,
            'image_id': item['image_id'],
        }

def rocov2_collate(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'captions': [[b['caption']] for b in batch],
        'image_id': [b['image_id'] for b in batch],
    }

g = torch.Generator()
g.manual_seed(0)
rocov2_train_loader = DataLoader(
    ROCOv2Dataset(rocov2_raw['train'], BLIP_PROCESSOR, CFG['max_length']),
    batch_size=CFG['batch_size'], shuffle=True,
    num_workers=CFG['num_workers'], pin_memory=True,
    collate_fn=rocov2_collate, worker_init_fn=worker_init_fn, generator=g)
rocov2_val_loader = DataLoader(
    ROCOv2Dataset(rocov2_raw['validation'], BLIP_PROCESSOR, CFG['max_length']),
    batch_size=CFG['batch_size'], shuffle=False,
    num_workers=CFG['num_workers'], pin_memory=True,
    collate_fn=rocov2_collate)

print(f'Train batches: {len(rocov2_train_loader)}  Val batches: {len(rocov2_val_loader)}')

sb = next(iter(rocov2_val_loader))
print(f"Smoke test passed. Sample image_id: {sb['image_id'][0]}")
print(f"Sample caption: {sb['captions'][0]}")

## 8. Evaluate function and checkpoint helpers (verbatim from your original notebook)

In [ ]:
@torch.no_grad()
def evaluate_blip_rocov2(model, loader, processor, cfg):
    model.eval()
    predictions, references = [], []
    for batch in loader:
        gen_ids = model.generate(
            pixel_values=batch['pixel_values'].to(DEVICE),
            max_length=cfg['max_length'], num_beams=cfg['beam_size'])
        predictions.extend(processor.batch_decode(gen_ids, skip_special_tokens=True))
        references.extend(batch['captions'])
    bleu4 = compute_bleu4(predictions, references)
    cider = compute_cider(predictions, references)
    meteor = compute_meteor(predictions, references)
    return bleu4, cider, meteor, predictions, references

def check_disk_space(min_gb=2.0):
    free_gb = _shutil.disk_usage(OUT).free / 1e9
    if free_gb < min_gb:
        print(f'  !! DISK WARNING: only {free_gb:.2f} GB free.')
        return False
    return True

def save_checkpoint(model, filename):
    if not check_disk_space():
        return None
    path = os.path.join(OUT, filename)
    try:
        torch.save(model.state_dict(), path)
        return path
    except RuntimeError as e:
        print(f'  !! Checkpoint save failed ({filename}): {e}')
        return None

def delete_checkpoint(filename):
    path = os.path.join(OUT, filename)
    if os.path.exists(path):
        os.remove(path)

print('Evaluate + checkpoint helpers defined.')

## 9. NEW: Joint-Schedule FT runner for ROCOv2

Structurally identical to `run_rocov2_damf` from your original notebook, minus the freeze/unfreeze at Stage 1 -- all parameters stay trainable across both phases.

In [ ]:
def run_rocov2_joint_schedule(seed):
    """Joint-Schedule FT for ROCOv2: same two-phase LR schedule as
    Staged FT (5e-5 for 2 epochs, then 1e-5 for 3 epochs) but with
    ALL parameters trainable throughout -- no freezing.
    """
    exp_name = f'joint_schedule_ft_rocov2_seed{seed}'
    json_path = os.path.join(OUT, f'{exp_name}.json')
    total_epochs = CFG['stage1_epochs'] + CFG['stage2_epochs']

    if os.path.exists(json_path):
        with open(json_path) as f:
            logs = json.load(f)
        done = len(logs['bleu4_per_epoch'])
        if done >= total_epochs:
            print(f'  [{exp_name}] already complete. Skipping.')
            return logs
        print(f'  [{exp_name}] resuming from epoch {done + 1}')
    else:
        logs = {
            'experiment': exp_name, 'dataset': 'ROCOv2', 'seed': seed,
            'method': 'joint_schedule_ft',
            'phase1_lr': CFG['lr_stage1'], 'phase1_epochs': CFG['stage1_epochs'],
            'phase2_lr': CFG['lr_stage2'], 'phase2_epochs': CFG['stage2_epochs'],
            'freeze_vision': False, 'freeze_language': False,
            'bleu4_per_epoch': [], 'cider_per_epoch': [], 'meteor_per_epoch': [],
            'train_loss_per_epoch': [], 'val_loss_per_epoch': [], 'rt_log': [],
        }
        done = 0

    seed_everything(seed)
    model = BlipForConditionalGeneration.from_pretrained(
        'Salesforce/blip-image-captioning-base').to(DEVICE)

    ckpt_path = os.path.join(OUT, f'{exp_name}_last.pt')
    if done > 0 and os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'  [{exp_name}] loaded checkpoint from epoch {done}')

    best_bleu4 = max(logs['bleu4_per_epoch']) if logs['bleu4_per_epoch'] else 0.0
    scaler = torch.amp.GradScaler('cuda')
    tracker = GradientTracker(model, 'blip')
    step_counter = [done * len(rocov2_train_loader)]

    if done < CFG['stage1_epochs']:
        opt1 = AdamW(model.parameters(), lr=CFG['lr_stage1'],
                    weight_decay=CFG['weight_decay'])
        for epoch in range(done + 1, CFG['stage1_epochs'] + 1):
            avg_loss, rt_log = train_one_epoch(model, rocov2_train_loader, opt1, scaler,
                                                tracker=tracker, step_counter=step_counter)
            val_loss = get_val_loss(model, rocov2_val_loader)
            bleu4, cider, meteor, preds, refs = evaluate_blip_rocov2(
                model, rocov2_val_loader, BLIP_PROCESSOR, CFG)
            logs['train_loss_per_epoch'].append(round(avg_loss, 4))
            logs['val_loss_per_epoch'].append(round(val_loss, 4))
            logs['bleu4_per_epoch'].append(round(bleu4, 4))
            logs['cider_per_epoch'].append(round(cider, 4) if cider else None)
            logs['meteor_per_epoch'].append(round(meteor, 4) if meteor else None)
            logs['rt_log'].extend(rt_log)
            if bleu4 > best_bleu4:
                best_bleu4 = bleu4
                logs['best_bleu4'] = bleu4
                logs['best_cider'] = cider
                logs['best_meteor'] = meteor
                logs['best_epoch'] = epoch
                logs['best_predictions'] = preds[:20]
                logs['best_references'] = refs[:20]
            cs = f'{cider:.4f}' if cider else 'N/A'
            ms = f'{meteor:.4f}' if meteor else 'N/A'
            print(f"  [{exp_name}] P1 epoch {epoch}/{CFG['stage1_epochs']} | "
                  f'train={avg_loss:.4f} val={val_loss:.4f} '
                  f'BLEU-4={bleu4:.4f} CIDEr={cs} METEOR={ms}')
            save_checkpoint(model, f'{exp_name}_last.pt')
            save_logs(logs, f'{exp_name}.json')
        done = CFG['stage1_epochs']

    opt2 = AdamW(model.parameters(), lr=CFG['lr_stage2'],
                weight_decay=CFG['weight_decay'])
    for epoch in range(max(done, CFG['stage1_epochs']) + 1, total_epochs + 1):
        avg_loss, rt_log = train_one_epoch(model, rocov2_train_loader, opt2, scaler,
                                            tracker=tracker, step_counter=step_counter)
        val_loss = get_val_loss(model, rocov2_val_loader)
        bleu4, cider, meteor, preds, refs = evaluate_blip_rocov2(
            model, rocov2_val_loader, BLIP_PROCESSOR, CFG)
        logs['train_loss_per_epoch'].append(round(avg_loss, 4))
        logs['val_loss_per_epoch'].append(round(val_loss, 4))
        logs['bleu4_per_epoch'].append(round(bleu4, 4))
        logs['cider_per_epoch'].append(round(cider, 4) if cider else None)
        logs['meteor_per_epoch'].append(round(meteor, 4) if meteor else None)
        logs['rt_log'].extend(rt_log)
        if bleu4 > best_bleu4:
            best_bleu4 = bleu4
            logs['best_bleu4'] = bleu4
            logs['best_cider'] = cider
            logs['best_meteor'] = meteor
            logs['best_epoch'] = epoch
            logs['best_predictions'] = preds[:20]
            logs['best_references'] = refs[:20]
        cs = f'{cider:.4f}' if cider else 'N/A'
        ms = f'{meteor:.4f}' if meteor else 'N/A'
        print(f'  [{exp_name}] P2 epoch {epoch}/{total_epochs} | '
              f'train={avg_loss:.4f} val={val_loss:.4f} '
              f'BLEU-4={bleu4:.4f} CIDEr={cs} METEOR={ms}')
        save_checkpoint(model, f'{exp_name}_last.pt')
        save_logs(logs, f'{exp_name}.json')

    tracker.remove()
    del model
    gc.collect()
    torch.cuda.empty_cache()
    delete_checkpoint(f'{exp_name}_last.pt')
    print(f'  [{exp_name}] complete. Best BLEU-4: {best_bleu4:.4f}\n')
    return logs

print('run_rocov2_joint_schedule defined.')

## 10. LAUNCH -- all 3 seeds

Resume-safe. This is the slowest of the new experiments (largest dataset) -- expect to let this run across multiple background sessions. Re-running the cell picks up exactly where each seed left off.

In [ ]:
print('=' * 60)
print('JOINT-SCHEDULE FT -- ROCOv2, all 3 seeds')
print('=' * 60)

results = {}
for seed in SEEDS:
    print(f"\n{'#'*60}\n# SEED {seed}\n{'#'*60}")
    results[seed] = run_rocov2_joint_schedule(seed)

print('\n' + '=' * 60)
print('ALL SEEDS COMPLETE')
print('=' * 60)
for seed, logs in results.items():
    rt = logs.get('rt_log', [])
    print(f"seed {seed}: best_bleu4={logs.get('best_bleu4'):.4f}  "
          f'rt_log entries={len(rt)}')
    if rt:
        print(f'   first: step {rt[0][0]}, Rt {rt[0][1]:.3f}  '
              f'last: step {rt[-1][0]}, Rt {rt[-1][1]:.3f}')

print(f'\nSaved to {OUT}/joint_schedule_ft_rocov2_seed*.json')
print('\nNOTE FOR THE PAPER: original ROCOv2 training (DAMF, Low-LR FT, etc.)')
print('used an unpinned transformers/peft install; the exact versions used')
print('are not recoverable. This new Joint-Schedule FT experiment (and the')
print('RMS-Balanced FT and Multimodal LoRA experiments to follow) are pinned')
print('to transformers==4.41.2, peft==0.11.1 -- the same versions validated')
print('for UICD and RSICD. Flag this as a documented limitation, not a silent')
print('assumption, when writing up these results.')